# 01 — Download Statistics Canada data

Download the source tables via the **StatCan Web Data Service (WDS)** REST API and
save raw CSVs + metadata + provenance JSON into `data/raw/<domain>/`.

This must be run on a **networked machine** (the WDS endpoints are not reachable
from the sandboxed analysis environment). The pull logic lives in
`src/statcan_api.py`; this notebook drives it and documents what was retrieved.

> **Governance reminder.** This dashboard produces a *triage* signal for human policy review. It does **not** determine social-assistance need, eligibility, or benefits. Claude outputs are claims, not facts. Missing or suppressed values are flagged, never invented. Any policy interpretation is reviewed by the AI Council before it is treated as decision-ready.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # import project modules
import utils
print("Project root:", utils.ROOT)
print("Raw data:    ", utils.RAW)
print("Processed:   ", utils.PROCESSED)
print("Outputs:     ", utils.OUTPUTS)
utils.ensure_dirs()

In [ ]:
import statcan_api
statcan_api.list_tables()  # key, PID, frequency, NEW flag, title

## 1. Freshness check — which tables changed?

LFS and CPI update monthly; income/low-income/population/wages update annually; the Census table is one-time. Only re-pull what changed.

In [ ]:
# statcan_api.check()   # uncomment on a networked machine

## 2. Confirm metadata for the three newly requested tables

`getCubeMetadata` returns the authoritative `cubeTitleEn`, dimensions, member codes, geography, and date range. Confirm these and update `data/metadata/data_sources.md` if a label changed.

In [ ]:
# for pid in (9810059701, 1410041701, 1410042601):
#     statcan_api.show_metadata(pid)

## 3. Download the raw tables

Each pull writes `data/raw/<domain>/<pid>.csv`, `<pid>_metadata.csv`, and `<pid>.source.json` (zip URL, table page, retrieval date).

In [ ]:
# Pull the three new tables:
# for key in ("census_income", "wages_occupation_annual", "wages_occupation_monthly"):
#     statcan_api.pull_one(key)

# Or pull everything configured:
# statcan_api.pull_one("labour_force")   # the LFS spine
# statcan_api.pull_all = lambda: [statcan_api.pull_one(k) for k in statcan_api.TABLES]

### Already present

The LFS table (14-10-0287-01) has already been retrieved and lives at `data/raw/labour_force/lfs_14100287_tidy.csv` (rate indicators, 1976-01 → 2026-05). The other tables download into their domain folders when the cells above are run.

In [ ]:
# Inventory what raw data is present right now
import os
for domain in ("labour_force","income","demographics","affordability"):
    d = utils.RAW / domain
    files = [f for f in os.listdir(d) if not f.startswith(".")] if d.exists() else []
    print(f"data/raw/{domain}: {files or '(empty — run the pull cells)'}")